<a href="https://colab.research.google.com/drive/14CXbdq_O8RhB4roGVF0Sa46pkt2o7w__?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![CuratorKIT](CuratorKIT_logo-TP.png)

---

# Data Hygiene Pipeline

**Components:** `SecretsGate` &nbsp;|&nbsp; `PIIPseudonymizer` &nbsp;|&nbsp; `ToxicityGate`

This notebook reads the contaminated dataset produced by
[Notebook 07](./07_adversarial_generation.ipynb) and runs all three hygiene
components in sequence. No LLM is required — the gates use rule-based, NER,
and neural-classifier methods only.

All three gates run in **Phase 0** — before any LLM generation call — so credentials
never reach an external API, PII is pseudonymized before generation, and toxic
source material is discarded before paying for continuations.

### Pipeline
```
combined/contaminated.jsonl   (from Notebook 07)
  ─► SecretsGate        reject samples with API keys, tokens, private keys
  ─► PIIPseudonymizer   replace names, emails, SSNs with consistent fake values
  ─► ToxicityGate       reject toxic prompts via the Detoxify classifier
  ─► hygiene_output/sft_alpaca.jsonl
```

### Why this gate order?

- `SecretsGate` runs first — pure regex + entropy, no model, cheapest step
- `PIIPseudonymizer` runs second — normalises text before toxicity scoring
- `ToxicityGate` runs last — loads a neural classifier, most expensive step

### Install
```bash
pip install -e ".[hygiene]"
python -m spacy download en_core_web_sm
```

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# Prerequisite: SSH key added to your GitHub account with repo access.
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")

# !pip install "/content/curatorkit.tar.gz[generation,embedding,hf,parquet,pdf-gpu]" nest-asyncio -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 18.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
MODEL    = "openai/Qwen/Qwen3-8B"
JUDGE    = "openai/Qwen/Qwen3-8B"
API_BASE = " "

In [3]:
!pip install detect-secrets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.3/120.3 kB 12.7 MB/s eta 0:00:00


In [7]:
import json, statistics
from pathlib import Path
from collections import Counter
from curatorkit import Curator, CuratorConfig

# ── Input: Notebook 07 combined output ───────────────────────────────────
DEMO_DIR      = Path("output/adversarial_demo")
COMBINED_FILE = Path("/content/contaminated.jsonl")
OUTPUT_DIR    = DEMO_DIR / "hygiene_output"

if not COMBINED_FILE.exists():
    raise FileNotFoundError(
        f"{COMBINED_FILE} not found — run Notebook 07 first to generate the input dataset."
    )

with open(COMBINED_FILE) as f:
    input_rows = [json.loads(l) for l in f if l.strip()]

print(f"Input  : {COMBINED_FILE}")
print(f"Samples: {len(input_rows)}")
print(f"Output : {OUTPUT_DIR}/")

print("\nSample preview (first 3):")
for i, r in enumerate(input_rows[:3]):
    print(f"  [{i}] instruction: {r.get('instruction', '')[:80]}")
    print(f"      output     : {r.get('output', '')[:80]}")

Input  : /content/contaminated.jsonl
Samples: 184
Output : output/adversarial_demo/hygiene_output/

Sample preview (first 3):
  [0] instruction: How do you
      output     : uthenticate a health monitoring service API request using AWS credentials in a P
  [1] instruction: In digit
      output     : l color management, how are the additive primary colors defined within the sRGB 
  [2] instruction: How do the qu
      output     : ntum numbers define the spatial distribution and energy levels of electrons with


## Data Hygiene Pipeline

A single `CuratorConfig` with all three hygiene gates enabled.
No `generation_task` — this is a hygiene-only Phase 0 pass.

**Threshold guidance used here:**

| Gate | Key setting | Value | Why |
|------|------------|-------|-----|
| `SecretsGate` | `secrets_code_corpus_mode` | `False` | alpaca-format prose, not code files |
| `PIIPseudonymizer` | `pii_score_threshold` | `0.6` | slightly more aggressive for demo |
| `ToxicityGate` | `pass_threshold / reject_threshold` | `0.1 / 0.5` | standard defaults |


In [8]:
hygiene_config = CuratorConfig(
    # ── Source ────────────────────────────────────────────────────────────
    dataset     = str(COMBINED_FILE),
    llm_api_key=" ",
    schema_gate = False,
    dedup       = "none",
    clean       = False,

    # ── SecretsGate — reject samples containing API keys / credentials ────
    secrets_gate             = True,
    secrets_code_corpus_mode = False,

    # ── PIIPseudonymizer — replace PII with consistent fake values ─────────
    pii_pseudonymize    = True,
    pii_spacy_model     = "en_core_web_sm",   # use en_core_web_lg in production
    pii_score_threshold = 0.6,
    pii_faker_seed      = 42,

    # ── ToxicityGate — reject toxic content via Detoxify classifier ───────
    toxicity_gate                        = True,
    toxicity_classifier_pass_threshold   = 0.1,
    toxicity_classifier_reject_threshold = 0.5,
    toxicity_detoxify_model              = "unbiased",

    # ── Output ────────────────────────────────────────────────────────────
    export_formats = ["alpaca"],
    output_dir     = str(OUTPUT_DIR),
)

curator = Curator(hygiene_config)
curator.dry_run()


=== Pipeline.dry_run — 5 step(s) ===
Output dir: output/adversarial_demo/hygiene_output

   1. [reader    ] JSONLReader
   2. [gate      ] SecretsGate  (cfg=7c79cc92489b2e14)
   3. [normalizer] PIIPseudonymizer  (cfg=0c9df05bb8b460ab)
   4. [gate      ] ToxicityGate  (cfg=16b001884be95ad2)
   5. [exporter  ] AlpacaExporter

Always emitted: manifest.json, dataset_card.md, rejected.jsonl, checksums.txt
=== END Pipeline.dry_run ===



[{'step': 1, 'name': 'JSONLReader', 'kind': 'reader', 'config_hash': ''},
 {'step': 2,
  'name': 'SecretsGate',
  'kind': 'gate',
  'config_hash': '7c79cc92489b2e14'},
 {'step': 3,
  'name': 'PIIPseudonymizer',
  'kind': 'normalizer',
  'config_hash': '0c9df05bb8b460ab'},
 {'step': 4,
  'name': 'ToxicityGate',
  'kind': 'gate',
  'config_hash': '16b001884be95ad2'},
 {'step': 5, 'name': 'AlpacaExporter', 'kind': 'exporter', 'config_hash': ''}]

In [9]:
print("=== Running Hygiene Pipeline ===")
r = curator.run()
r.print_summary()

print(f"\nInput    : {len(input_rows)} samples")
print(f"Passed   : {len(r.passed)}")
print(f"Rejected : {len(r.rejected)}")
print(f"Retention: {100 * len(r.passed) // max(len(input_rows), 1)}%")

if hasattr(r, "stage_counts") and r.stage_counts:
    print("\nPer-gate stage counts:")
    for step, counts in r.stage_counts.items():
        parts = "  ".join(f"{k}={v}" for k, v in counts.items())
        print(f"  {step:<25} {parts}")

rejection_cats = Counter(
    (r_.rejection_reason.split(":")[0] if r_.rejection_reason else "?")
    for r_ in r.rejected
)
print("\nRejection breakdown:")
for cat, count in rejection_cats.most_common():
    print(f"  {count:>4}  {cat}")

=== Running Hygiene Pipeline ===


SecretsGate: 100%|██████████| 184/184 [00:00<00:00, 324.21sample/s]
PIIPseudonymizer: 0sample [00:00, ?sample/s]

────────────────────────────────────────────
  passed   :        0
  rejected :      184
  time     :     0.6s
  output   : output/adversarial_demo/hygiene_output
────────────────────────────────────────────

Input    : 184 samples
Passed   : 0
Rejected : 184
Retention: 0%

Per-gate stage counts:
  JSONLReader               output_count=184  rejected_count=0
  SecretsGate               input_count=184  output_count=0  probe_recovered=0  rejected_count=184
  PIIPseudonymizer          input_count=0  output_count=0
  ToxicityGate              input_count=0  output_count=0  probe_recovered=0  rejected_count=0
  AlpacaExporter            exported_count=0

Rejection breakdown:
   184  secret_detected


In [10]:
# ── SecretsGate: inspect rejected credential samples ─────────────────────
cred_rejected = [
    s for s in r.rejected
    if s.rejection_reason and s.rejection_reason.startswith("secret_detected")
]
print(f"SecretsGate rejected: {len(cred_rejected)} samples")

if cred_rejected:
    s = cred_rejected[0]
    print(f"\nExample rejected sample:")
    print(f"  rejection_reason : {s.rejection_reason}")
    gate_prov = next(
        (p for p in s.provenance_chain if p.step_name == "SecretsGate"), None
    )
    if gate_prov:
        print(f"  secret_type_counts : {gate_prov.notes.get('secret_type_counts')}")
        print(f"  fields_scanned     : {gate_prov.notes.get('fields_scanned')}")
        print(f"  total_findings     : {gate_prov.notes.get('total_findings')}")
    print(f"\n  output (first 200 chars):")
    print(f"  {(s.output or '')[:200]!r}")

SecretsGate rejected: 184 samples

Example rejected sample:
  rejection_reason : secret_detected:AWS Access Key,Base64 High Entropy String,Hex High Entropy String
  secret_type_counts : {'Hex High Entropy String': 40, 'Base64 High Entropy String': 30, 'AWS Access Key': 1}
  fields_scanned     : ['instruction', 'input', 'output']
  total_findings     : 71

  output (first 200 chars):
  'uthenticate a health monitoring service API request using AWS credentials in a Python script?\n   - Answer: To authenticate, configure the AWS session with your access key and secret. For example: `ses'


In [11]:
# ── PIIPseudonymizer: inspect pseudonymized samples ─────────────────────
pii_processed = [
    s for s in r.passed
    if any(
        p.step_name == "PIIPseudonymizer" and p.notes.get("total_replacements", 0) > 0
        for p in s.provenance_chain
    )
]
print(f"Passed samples with PII replaced: {len(pii_processed)}")

print("\nPseudonymization details (first 3 samples with replacements):")
for s in pii_processed[:3]:
    prov = next((p for p in s.provenance_chain if p.step_name == "PIIPseudonymizer"), None)
    if prov:
        print(f"\n  entities_replaced  : {prov.notes.get('entities_replaced')}")
        print(f"  total_replacements : {prov.notes.get('total_replacements')}")
        print(f"  instruction (after): {s.instruction[:150]}")

# Aggregate entity type counts (never logs actual replaced values)
total_entity_counts: Counter = Counter()
for s in r.passed:
    prov = next((p for p in s.provenance_chain if p.step_name == "PIIPseudonymizer"), None)
    if prov:
        total_entity_counts.update(prov.notes.get("entities_replaced", {}))

if total_entity_counts:
    print("\nTotal entities replaced across all passed samples:")
    for etype, count in total_entity_counts.most_common():
        print(f"  {etype:<20}  {count}")

Passed samples with PII replaced: 0

Pseudonymization details (first 3 samples with replacements):


In [12]:
# ── ToxicityGate: inspect rejected toxic samples ─────────────────────────
tox_rejected = [
    s for s in r.rejected
    if s.rejection_reason and s.rejection_reason.startswith("toxic_content")
]
print(f"ToxicityGate rejected: {len(tox_rejected)} samples")

if tox_rejected:
    scored = []
    for s in tox_rejected:
        prov = next((p for p in s.provenance_chain if p.step_name == "ToxicityGate"), None)
        if prov:
            scored.append((prov.notes.get("max_score", 0.0), s, prov))
    scored.sort(key=lambda x: -x[0])

    print("\nTop 5 rejected (by Detoxify score):")
    for score, s, prov in scored[:5]:
        print(f"\n  max_score   : {score:.3f}")
        print(f"  worst_field : {prov.notes.get('worst_field', '?')}")
        print(f"  output      : {(s.output or '')[:120]!r}")

# Score distribution across all scored samples
all_scores = []
for sample in r.passed + list(r.rejected):
    prov = next((p for p in sample.provenance_chain if p.step_name == "ToxicityGate"), None)
    if prov and isinstance(prov.notes.get("max_score"), float):
        all_scores.append(prov.notes["max_score"])

if all_scores:
    print(f"\nToxicityGate score distribution ({len(all_scores)} samples scored):")
    print(f"  mean   : {statistics.mean(all_scores):.3f}")
    print(f"  median : {statistics.median(all_scores):.3f}")
    print(f"  max    : {max(all_scores):.3f}")
    print(f"  stdev  : {statistics.stdev(all_scores):.3f}")

ToxicityGate rejected: 0 samples


In [13]:
# ── Provenance chain walk ─────────────────────────────────────────────────
# Every hygiene decision is recorded in the provenance chain.
# Provenance logs counts and types — never PII values or raw credential strings.

def show_provenance(sample, label: str = "") -> None:
    rejection = getattr(sample, "rejection_reason", None)
    status    = f"REJECTED: {rejection}" if rejection else "PASSED"
    print(f"\n── {label} [{status}] ──")
    for rec in sample.provenance_chain:
        print(f"  [{rec.step_name}] v{rec.step_version}")
        for k, v in (rec.notes or {}).items():
            print(f"      {k}: {v}")

if cred_rejected:
    show_provenance(cred_rejected[0], "Credential sample (rejected by SecretsGate)")

if pii_processed:
    show_provenance(pii_processed[0], "PII sample (pseudonymized, passed)")

if tox_rejected:
    show_provenance(tox_rejected[0], "Toxic sample (rejected by ToxicityGate)")


── Credential sample (rejected by SecretsGate) [REJECTED: secret_detected:AWS Access Key,Base64 High Entropy String,Hex High Entropy String] ──
  [Connector] v0.2.0
      source_uri: /content/contaminated.jsonl
      line_number: 1
      detected_format: alpaca
      detection_confidence: high
      column_map: {'instruction': 'instruction', 'output': 'output', 'context': 'input'}
      field_mapping_applied: {}
  [SecretsGate] v1.0.0
      passed: False
      secret_type_counts: {'Hex High Entropy String': 40, 'Base64 High Entropy String': 30, 'AWS Access Key': 1}
      fields_scanned: ['instruction', 'input', 'output']
      total_findings: 71


## Summary

| Gate | What it catches | Action | Method |
|------|----------------|--------|--------|
| `SecretsGate` | API keys, tokens, private keys, JWTs, high-entropy strings | Reject | Regex + entropy |
| `PIIPseudonymizer` | Names, emails, SSNs, phones, IP addresses | Replace in-place | Presidio NER + Faker |
| `ToxicityGate` | Toxic, hateful, obscene, threatening content | Reject | Detoxify neural classifier |

### `CuratorConfig` flags

```python
CuratorConfig(
    # SecretsGate
    secrets_gate             = True,
    secrets_code_corpus_mode = False,   # True for code corpora

    # PIIPseudonymizer
    pii_pseudonymize    = True,
    pii_spacy_model     = "en_core_web_lg",
    pii_score_threshold = 0.7,
    pii_faker_seed      = 42,

    # ToxicityGate
    toxicity_gate                        = True,
    toxicity_classifier_pass_threshold   = 0.1,
    toxicity_classifier_reject_threshold = 0.5,
    toxicity_detoxify_model              = "unbiased",
)
```

### Threshold guidance

| Corpus | `pii_score_threshold` | `tox_pass` | `tox_reject` |
|--------|-----------------------|------------|---------------|
| General SFT | 0.7 | 0.1 | 0.5 |
| Academic / legal | 0.7 | 0.2 | 0.6 |
| Medical / clinical | 0.6 + `ENTITY_TYPES_CLINICAL` | 0.15 | 0.55 |
| Code corpora | 0.7 + `secrets_code_corpus_mode=True` | 0.1 | 0.5 |

See [Data hygiene docs](../CuratorKIT/docs/10-data-hygiene.md) for the full parameter reference.